In [2]:
import os
import torch
from PIL import Image
from sklearn.metrics import confusion_matrix

import numpy as np

In [3]:
# Function to load masks from a folder with a specific file extension
def load_masks_from_folder(folder_path, file_extension='.tif'):
    masks = []
    for filename in os.listdir(folder_path):
        if filename.endswith(file_extension):
            mask_path = os.path.join(folder_path, filename)
            mask = Image.open(mask_path).convert('L')  # Assuming masks are single-channel (grayscale)
            
            # Convert to NumPy array and then to PyTorch tensor
            mask = torch.tensor(np.array(mask)).unsqueeze(0).float()
            
            masks.append((filename, mask))
    return masks

In [4]:
# Paths to your folders containing reference and predicted masks
reference_folder = r'D:\deep_learning\Sentinel_forestfire\MY_TEST_SENTINEL_IMAGERY_DATA\imagery_data_complete_0828\erdas_isodata\1113\results\reference2'
predicted_folder = r'D:\deep_learning\Sentinel_forestfire\MY_TEST_SENTINEL_IMAGERY_DATA\imagery_data_complete_0828\erdas_isodata\1113\results\isodata_sample3'

In [5]:
# Load reference masks with the '.tif' extension
reference_masks = load_masks_from_folder(reference_folder, file_extension='.tif')

# Load predicted masks with the '.tif' extension
predicted_masks = load_masks_from_folder(predicted_folder, file_extension='.tif')

# Ensure the number of masks is the same in both folders
assert len(reference_masks) == len(predicted_masks), "Number of masks does not match."

# Initialize lists to store evaluation metrics
oas, precisions, recalls, f1s, kappas, ious = [], [], [], [], [], []

In [6]:
# Iterate over each pair of masks
for (reference_filename, reference_mask), (predicted_filename, predicted_mask) in zip(reference_masks, predicted_masks):
    # Convert masks to numpy arrays
    reference_mask_np = reference_mask.cpu().numpy().flatten()
    predicted_mask_np = predicted_mask.cpu().numpy().flatten()

    # Calculate confusion matrix
    cm = confusion_matrix(reference_mask_np, predicted_mask_np, labels=[0, 1])

    # Calculate evaluation metrics
    tn, fp, fn, tp = cm.ravel()

    # Overall Accuracy (OA)
    oa = (tp + tn) / (tp + tn + fp + fn)

    # Precision
    precision = tp / (tp + fp)

    # Recall
    recall = tp / (tp + fn)

    # F1 Score
    f1 = 2 * (precision * recall) / (precision + recall)

    # Kappa Score
    po = (tp + tn) / (tp + tn + fp + fn)
    pe = ((tp + fp) * (tp + fn) + (tn + fp) * (tn + fn)) / ((tp + tn + fp + fn) ** 2)
    kappa = (po - pe) / (1 - pe)

    # Intersection over Union (IoU) Score
    iou = tp / (tp + fp + fn)

    # Append scores to lists
    oas.append(oa)
    precisions.append(precision)
    recalls.append(recall)
    f1s.append(f1)
    kappas.append(kappa)
    ious.append(iou)

    # Log the scores for the current image pair
    print(f"Image Pair Metrics for Reference Mask '{reference_filename}' and Predicted Mask '{predicted_filename}':")
    print(f"Overall Accuracy (OA): {oa}")
    print(f"Precision: {precision}")
    print(f"Recall: {recall}")
    print(f"F1 Score: {f1}")
    print(f"Kappa Score: {kappa}")
    print(f"IoU Score: {iou}")
    print("\n")

# Calculate average metrics
average_oa = sum(oas) / len(oas)
average_precision = sum(precisions) / len(precisions)
average_recall = sum(recalls) / len(recalls)
average_f1 = sum(f1s) / len(f1s)
average_kappa = sum(kappas) / len(kappas)
average_iou = sum(ious) / len(ious)

# Print or log the average results
print(f"Average Overall Accuracy (OA): {average_oa}")
print(f"Average Precision: {average_precision}")
print(f"Average Recall: {average_recall}")
print(f"Average F1 Score: {average_f1}")
print(f"Average Kappa Score: {average_kappa}")
print(f"Average IoU Score: {average_iou}")

Image Pair Metrics for Reference Mask 'T52SDH_20180331T020649_2018021_mask.tif' and Predicted Mask '20180331_2018021_final_burned_unburned.tif':
Overall Accuracy (OA): 0.9681968688964844
Precision: 0.8664787556112329
Recall: 0.8469387755102041
F1 Score: 0.8565973476443572
Kappa Score: 0.8387152010027987
IoU Score: 0.749165087101724


Image Pair Metrics for Reference Mask 'T52SDH_20180331T020649_2018021_mask2.tif' and Predicted Mask '20180331_2018021_final_burned_unburned_0.5_rule.tif':
Overall Accuracy (OA): 0.9586296081542969
Precision: 0.8144808650554218
Recall: 0.8172789115646258
F1 Score: 0.8158774893465306
Kappa Score: 0.7925743801574407
IoU Score: 0.6890144237662374


Average Overall Accuracy (OA): 0.9634132385253906
Average Precision: 0.8404798103333273
Average Recall: 0.832108843537415
Average F1 Score: 0.836237418495444
Average Kappa Score: 0.8156447905801197
Average IoU Score: 0.7190897554339808
